# MoE的复现


In [22]:
import math
import torch
from torch import nn
from transformers.activations import ACT2FN
import torch.nn.functional as F
from transformers import PretrainedConfig

In [23]:
class Config(PretrainedConfig):
    """MoE模型的配置类，继承自Hugging Face的PretrainedConfig"""
    
    def __init__(
            self,
            dropout: float = 0.0,
            hidden_act: str = 'silu',
            hidden_size: int = 512,
            intermediate_size: int = None,
            num_experts_per_tok: int = 2,
            n_routed_experts: int = 4,
            n_shared_experts: int = 1,
            scoring_func: str = 'softmax',
            aux_loss_alpha: float = 0.1,
            seq_aux: bool = True,
            norm_topk_prob: bool = True,
            **kwargs
    ):
        """
        初始化MoE配置
        
        Args:
            dropout: Dropout率
            hidden_act: 激活函数类型
            hidden_size: 隐藏层维度
            intermediate_size: 中间层维度，如为None则自动计算
            num_experts_per_tok: 每个token选择的专家数量
            n_routed_experts: 总的专家数量
            n_shared_experts: 共享专家数量
            scoring_func: 评分函数类型，默认为'softmax'
            aux_loss_alpha: 辅助损失的alpha参数
            seq_aux: 是否在序列级别计算辅助损失
            norm_topk_prob: 是否标准化top-k概率
        """
        super().__init__(**kwargs)
        self.dropout = dropout
        self.hidden_act = hidden_act
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts_per_tok = num_experts_per_tok    
        self.n_routed_experts = n_routed_experts          
        self.n_shared_experts = n_shared_experts          
        self.scoring_func = scoring_func                  
        self.aux_loss_alpha = aux_loss_alpha              
        self.seq_aux = seq_aux                            
        self.norm_topk_prob = norm_topk_prob

## 单个专家的结构  
一般先是一个上采样再加一个下采样

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config:Config):
        super().__init__()
        if config.intermediate_size == None:
            intermediate_size= int(config.hidden_size*8/3)
            config.intermediate_size = 64*((intermediate_size+63)//64)

        self.gate_proj = nn.Linear(config.hidden_size,config.intermediate_size,bias=False)
        self.up_proj = nn.Linear(config.hidden_size,config.intermediate_size,bias=False)
        self.down_proj = nn.Linear(config.intermediate_size,config.hidden_size,bias=False)
        self.dropout = nn.Dropout(config.dropout)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self,x):
        return self.dropout(self.down_proj(self.up_proj(x)*self.act_fn(self.gate_proj(x))))

## MoE的门控路由器

In [ ]:
class MoEGate(nn.Module):
        def __init__(self, config: Config):
            super().__init__()
            self.config = config
            self.top_k = config.num_experts_per_tok  
            self.n_routed_experts = config.n_routed_experts  
            self.scoring_func = config.scoring_func                      
            self.alpha = config.aux_loss_alpha       
            self.seq_aux = config.seq_aux           
            self.norm_topk_prob = config.norm_topk_prob  
            self.gating_dim = config.hidden_size          
            self.weight = nn.Parameter(torch.empty((self.n_routed_experts, self.gating_dim)))                    
            self.reset_parameters()
        # kaiming初始化参数，为ReLU族激活函数设计好的“刚好合适”的初始化
        def reset_parameters(self)->None:
            import torch.nn.init as init
            init.kaiming_uniform_(self.weight,a=math.sqrt(5))

        def forward(self, hidden_sates):
            bsz, seq_len, h = hidden_sates.shape
            hidden_sates = hidden_sates.view(-1,h)

            logits = F.linear(hidden_sates, self.weight )

            if self.scoring_func == "softmax":
                scores = logits.softmax(dim=-1)
            else:
                raise NotImplementedError(f"对于MoE gating，这个函数并不合适：{self.scoring_func}")

            topk_weight, topk_idx = torch.topk(scores, k=self.top_k, dim=-1, sorted=False)

            

        
              